<a href="https://colab.research.google.com/github/rawanmd/segmentation_coverless_steg/blob/adaptive_window/Segmentation_Coverless_Steg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from textwrap import indent
from google.colab import drive

import os
import glob
import json

## Connect to Drive

In [2]:
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# Dataset root
dataset_dir = "/content/drive/MyDrive/coco_dataset"
os.makedirs(dataset_dir, exist_ok=True)

# Move into dataset folder
%cd $dataset_dir

# Create desired folders
os.makedirs("annotations", exist_ok=True)
os.makedirs("train", exist_ok=True)
os.makedirs("val", exist_ok=True)

# ==========================
# 1. ANNOTATIONS
# ==========================
# !wget -c http://images.cocodataset.org/annotations/annotations_trainval2017.zip

# !unzip -q annotations_trainval2017.zip
# !rm annotations_trainval2017.zip

# # ==========================
# # 2. VALIDATION IMAGES
# # ==========================
# !wget -c http://images.cocodataset.org/zips/val2017.zip

# !unzip -q val2017.zip
# !rm val2017.zip

# # Move images into val/
# !mv val2017/* val/
# !rmdir val2017

# ==========================
# 3. TRAIN IMAGES
# ==========================
# !wget -c http://images.cocodataset.org/zips/train2017.zip

# !unzip -q train2017.zip
# !rm train2017.zip

# # Move images into train/
# !mv train2017/* train/
# !rmdir train2017

Mounted at /content/drive
/content/drive/MyDrive/coco_dataset


## Import SAM

In [3]:
!pip install git+https://github.com/facebookresearch/segment-anything.git
!pip install opencv-python pycocotools matplotlib
!pip install torch torchvision

import torch
import cv2
import matplotlib.pyplot as plt

!wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-3t1woizo
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-3t1woizo
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Preparing metadata (setup.py) ... done
  Created wheel for segment_anything: filename=segment_anything-1.0-py3-none-any.whl size=36592 sha256=0ff9fd0b047c3b2576d90fa6dffa4d8e2c9ff73f9d45aa79c23ac37115871cbe
  Stored in directory: /tmp/pip-ephem-wheel-cache-kmqoe5ft/wheels/29/82/ff/04e2be9805a1cb48bec0b85b5a6da6b63f647645750a0e42d4
Successfully built segment_anything
--2026-08-11 14:30:41--  https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 65.9.168.62, 65.9.168.52, 65.9.168.81, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|65.9.168.62|

In [4]:
MODEL_TYPE = "vit_b"
CHECKPOINT = "sam_vit_b_01ec64.pth"

device = "cuda" if torch.cuda.is_available() else "cpu"

sam = sam_model_registry[MODEL_TYPE](checkpoint=CHECKPOINT)
sam.to(device=device)

mask_generator = SamAutomaticMaskGenerator(sam)

## Functions

In [5]:
def extract_masks(image_path):

  import math

  image = cv2.imread(image_path)
  image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
  masks = mask_generator.generate(image)

  binary_seq = ""

  for i in range(len(masks) - 1):
    bbox1 = masks[i]['bbox']
    x1, y1 = bbox1[:2]

    bbox2 = masks[i+1]['bbox']
    x2, y2 = bbox2[:2]

    d_squared = (x2 - x1)**2 + (y2 - y1)**2
    d = math.sqrt(d_squared)

    if(d <= 300):
      binary_seq += '0'
    else:
      binary_seq += '1'

  return binary_seq, len(binary_seq)

def get_image_paths(dir):
  image_paths = sorted(glob.glob(os.path.join(dir, '*.jpg')))
  print(f"Found {len(image_paths)} images.")
  return image_paths

def build_index(image_paths):
  index = {}

  for image_path in image_paths:
    # call segmentation model
    binary_seq, length = extract_masks(image_path)

    if binary_seq is None:
      continue

    if binary_seq not in index:
      index[binary_seq] = []

    index[binary_seq].append({"image path": image_path,
                              "length": length})

    with open(str(dataset_dir)+"/inverted_index.json", "w") as f:
      json.dump(index, f, indent=4)

  return index

## Pipeline

### Text Encryption

### Divide and match the message to images

In [10]:
PATH = "/content/drive/MyDrive/coco_dataset/train2017"

secret_msg = "Hello World"
ascii_secret_msg = [ord(char) for char in secret_msg]

# add text enc here

binary_msg = [f"{val:08b}" for val in ascii_secret_msg]
binary_msg = ''.join(binary_msg)

# build index if not already there
# index = build_index(PATH)

# load index if it exists
index = None
with open("/content/drive/MyDrive/coco_dataset/inverted_index.json", "r") as f:
  index = json.load(f)



In [15]:
for idx in index:
  print(idx)
  break

010001101010010


In [17]:
window_size = max(len(key) for key in index.keys())
print("Largest length:", window_size)

for i in range(0, len(binary_msg), window_size):
  chunk = binary_msg[i:i + window_size]
  key = ''

  for idx in index:
    if idx.startswith(chunk):
      key = idx
      break

  print(f"key: {key}")

Largest length: 309
key: 


In [ ]:
window_size = 10

prefix_map = {}

for code in index.keys():
    prefix = code[:window_size]

    if prefix not in prefix_map:
        prefix_map[prefix] = []

    prefix_map[prefix].append(code)

prefix_map["0110011010"]

['0110011010000011100000111011100001011010100000001100',
 '011001101011011000010010100110011000100001101101010100000001110110101101001']

In [ ]:
all_codes = list(index.keys())

print(index.keys())

prefix_map = {}

for window_size in range(10):
  for code in all_codes:
      # window_size = index[code][0]['length']
      prefix = code[:window_size]
      if prefix not in prefix_map:
          prefix_map[prefix] = []

      prefix_map[prefix].append(code)

prefix_map

dict_keys(['010001101010010', '10110010000000000000000000000', '01100101011010001000000000000000000000101010001001', '11110111011111101101111', '101001100000010010000000000', '1010110001000010000011000110000000000000001110010110000000001100010010110011100100000111110011000010110011100001011110000100010', '11011011111100110110110110001101000', '00011000101101111001101100011111110010000000110010101111001101011000000000100001110001', '000111011000001000110000101010100101100010000110100000110100010000', '0001010011110001011000011110011100111110', '001001101101000101101000100001101010011001001111001111111000111100101111100110001000000', '1110001100011010001110110010010000111100100000011001100000111111011010', '001100000100000100011100111000011000001100000100001100011011111011011000000000000001000001100', '01100000000000000000000', '000011110000100111000000111110010110101001001011110100010111000001100010110000011101101110', '11111110011111000000001100010011100000', '1100010000011010001100000

{'': ['010001101010010',
  '10110010000000000000000000000',
  '01100101011010001000000000000000000000101010001001',
  '11110111011111101101111',
  '101001100000010010000000000',
  '1010110001000010000011000110000000000000001110010110000000001100010010110011100100000111110011000010110011100001011110000100010',
  '11011011111100110110110110001101000',
  '00011000101101111001101100011111110010000000110010101111001101011000000000100001110001',
  '000111011000001000110000101010100101100010000110100000110100010000',
  '0001010011110001011000011110011100111110',
  '001001101101000101101000100001101010011001001111001111111000111100101111100110001000000',
  '1110001100011010001110110010010000111100100000011001100000111111011010',
  '001100000100000100011100111000011000001100000100001100011011111011011000000000000001000001100',
  '01100000000000000000000',
  '000011110000100111000000111110010110101001001011110100010111000001100010110000011101101110',
  '11111110011111000000001100010011100000',
 